# Challenge 03 — Connect Your Agents to Tools (Coach Solution)

This notebook is the **coach solution** for Challenge 03. It demonstrates all four tool types
using the **Azure AI Foundry SDK v2** (azure-ai-projects 2.0.x) with the conversations API.

## What's covered
- **Part 1:** Function Calling — define a tool schema, create an agent, handle the function call loop
- **Part 2:** File Search — upload files, create a vector store, query grounded data
- **Part 3:** Azure AI Search — connect to a search index for production-grade RAG
- **Part 4:** Grounding comparison — agent with tools vs. without tools

---
## Setup — Environment & Authentication

In [ ]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv
from azure.identity import AzureCliCredential, InteractiveBrowserCredential, ChainedTokenCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    PromptAgentDefinition,
    FunctionTool,
    CodeInterpreterTool,
    AzureAISearchTool,
    AzureAISearchToolResource,
    AISearchIndexResource,
    AzureAISearchQueryType,
)

# Load env vars from Student/Resources/.env
env_path = Path("__file__").resolve().parent.parent.parent / "Student" / "Resources" / ".env" if "__file__" in dir() else Path.cwd().parent.parent / "Student" / "Resources" / ".env"
load_dotenv(dotenv_path=env_path, override=True)

PROJECT_ENDPOINT = os.getenv("AZURE_AI_FOUNDRY_ENDPOINT")
MODEL_DEPLOYMENT_NAME = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")
TENANT_ID = os.getenv("AZURE_TENANT_ID")
AI_SEARCH_PROJECT_CONNECTION_ID = os.getenv("AI_SEARCH_PROJECT_CONNECTION_ID", "")
AI_SEARCH_INDEX_NAME = os.getenv("AI_SEARCH_INDEX_NAME", "")

assert PROJECT_ENDPOINT, "AZURE_AI_FOUNDRY_ENDPOINT is missing from .env"
assert MODEL_DEPLOYMENT_NAME, "AZURE_OPENAI_DEPLOYMENT_NAME is missing from .env"

print(f"Endpoint: {PROJECT_ENDPOINT}")
print(f"Model: {MODEL_DEPLOYMENT_NAME}")
print(f"Tenant: {TENANT_ID or '(not set)'}")
print(f"AI Search Connection: {AI_SEARCH_PROJECT_CONNECTION_ID or '(not set — Part 3 will be skipped)'}")

In [ ]:
# Authenticate and create the project client
credential = ChainedTokenCredential(
    AzureCliCredential(tenant_id=TENANT_ID) if TENANT_ID else AzureCliCredential(),
    InteractiveBrowserCredential(tenant_id=TENANT_ID) if TENANT_ID else InteractiveBrowserCredential(),
)

project_client = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=credential,
)

# Get an OpenAI client for conversations
openai_client = project_client.get_openai_client()

print("✅ Connected to Azure AI Foundry")

---
## Part 1: Function Calling

Define a function tool schema, create an agent with it, and handle the function calling loop.

In [ ]:
# Define the function tool using FunctionTool from the v2 SDK
get_news_tool = FunctionTool(
    name="get_news_articles",
    description="Retrieve recent news articles about a given topic, optionally filtered by geographic region.",
    parameters={
        "type": "object",
        "properties": {
            "topic": {
                "type": "string",
                "description": "The news topic to search for (e.g., 'travel advisories', 'airline strikes')"
            },
            "region": {
                "type": "string",
                "description": "Geographic region to filter results (e.g., 'Europe', 'Asia')"
            },
            "max_results": {
                "type": "integer",
                "description": "Number of articles to return (default 3)"
            }
        },
        "required": ["topic"],
        "additionalProperties": False
    },
    strict=True,
)

print("✅ Function tool schema defined")
print(f"   Name: {get_news_tool.name}")
print(f"   Parameters: {json.dumps(get_news_tool.parameters, indent=2)}")

In [ ]:
# Create the agent with the function tool
news_agent_with_tools = project_client.agents.create_version(
    agent_name="NewsAgentWithTools",
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT_NAME,
        instructions=(
            "You are a travel news assistant with access to a news retrieval tool. "
            "When a user asks about news, events, or travel advisories, ALWAYS use the "
            "get_news_articles tool to retrieve current information. "
            "Never fabricate news articles or events. "
            "Summarize the retrieved articles in a clear, organized format with headlines and key points. "
            "If the tool returns no results, clearly state that no recent news was found."
        ),
        tools=[get_news_tool],
    ),
)
print(f"✅ Agent created — name: {news_agent_with_tools.name}, version: {news_agent_with_tools.version}")

In [ ]:
# Simulated news data — in production this would call a real API
def get_news_articles(topic: str, region: str = None, max_results: int = 3) -> str:
    """Simulated news retrieval function."""
    articles = [
        {
            "headline": f"Major {topic} developments reported in {region or 'global'} markets",
            "summary": f"Experts highlight significant changes in {topic} affecting travelers and businesses alike.",
            "source": "Travel Wire",
            "date": "2026-06-10"
        },
        {
            "headline": f"New {topic} regulations announced for international travelers",
            "summary": f"Government officials released updated guidelines regarding {topic} that may impact travel plans.",
            "source": "Global News Daily",
            "date": "2026-06-09"
        },
        {
            "headline": f"{region or 'World'} summit addresses {topic} concerns",
            "summary": f"Leaders from multiple countries convened to discuss {topic} and its implications for tourism.",
            "source": "International Herald",
            "date": "2026-06-08"
        },
    ]
    return json.dumps(articles[:max_results])

print("✅ Simulated news function ready")
print("Sample output:")
print(get_news_articles("travel safety", "Europe", 2))

In [ ]:
# Create a conversation and invoke the agent
conversation = openai_client.conversations.create(
    items=[{"type": "message", "role": "user", "content": "What are the latest travel advisories for Europe?"}],
)
print(f"✅ Conversation created — ID: {conversation.id}")

# Call the agent — it may respond with a function_call in its output
response = openai_client.responses.create(
    conversation=conversation.id,
    extra_body={"agent_reference": {"name": news_agent_with_tools.name, "type": "agent_reference"}},
)

print(f"\nResponse status: {response.status}")
print(f"Output items: {len(response.output)}")
for item in response.output:
    print(f"  - type: {item.type}")

In [ ]:
# Handle the function calling loop
# Check if the response contains function_call items
function_calls = [item for item in response.output if item.type == "function_call"]

if function_calls:
    print(f"🔧 Agent wants to call {len(function_calls)} function(s)")
    
    # Build the input items: include the function calls + their outputs
    input_items = []
    for fc in function_calls:
        args = json.loads(fc.arguments)
        print(f"   Function: {fc.name}")
        print(f"   Arguments: {args}")
        
        # Execute the local function
        result = get_news_articles(**args)
        print(f"   Result: {result[:100]}...")
        
        # Add the function call output to send back
        input_items.append({
            "type": "function_call_output",
            "call_id": fc.call_id,
            "output": result,
        })
    
    # Send the function outputs back via conversations.items.create
    openai_client.conversations.items.create(
        conversation_id=conversation.id,
        items=input_items,
    )
    
    # Invoke the agent again to get the final grounded response
    final_response = openai_client.responses.create(
        conversation=conversation.id,
        extra_body={"agent_reference": {"name": news_agent_with_tools.name, "type": "agent_reference"}},
    )
    print(f"\n✅ Final response:")
    print(f"\n--- AGENT ---\n{final_response.output_text}")
else:
    # Agent answered directly (no function call needed)
    print(f"\n--- AGENT ---\n{response.output_text}")

---
## Part 2: File Search — Upload Files & Query Grounded Data

Upload the hotel/flight Excel file and create an agent that can search through it.

In [ ]:
# Upload the travel data file to a vector store
# The file is in Student/Resources/Challenge-03/travel-data.xlsx
data_file_path = Path.cwd().parent.parent / "Student" / "Resources" / "Challenge-03" / "travel-data.xlsx"
assert data_file_path.exists(), f"Data file not found at {data_file_path}"

# Upload via the OpenAI files API
with open(data_file_path, "rb") as f:
    uploaded_file = openai_client.files.create(file=f, purpose="assistants")
print(f"✅ File uploaded — ID: {uploaded_file.id}")

# Create a vector store and add the file
vector_store = openai_client.vector_stores.create(
    name="travel-data-store",
    file_ids=[uploaded_file.id],
)
print(f"✅ Vector store created — ID: {vector_store.id}")

In [ ]:
# Create an agent with File Search + Code Interpreter tools
travel_agent = project_client.agents.create_version(
    agent_name="TravelAgentWithSearch",
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT_NAME,
        instructions=(
            "You are a travel planning assistant with access to hotel and flight data files. "
            "Always search through the uploaded data to answer questions about hotels, flights, prices, and amenities. "
            "Cite specific data (hotel names, nightly rates, flight numbers, prices) from the files. "
            "Never make up hotel names, prices, or flight details. "
            "If the information is not in the files, clearly state that. "
            "When asked for cost comparisons or budget calculations, use the code interpreter to compute exact numbers."
        ),
        tools=[
            {"type": "file_search", "vector_store_ids": [vector_store.id]},
            CodeInterpreterTool(),
        ],
    ),
)
print(f"✅ Travel agent created — name: {travel_agent.name}, version: {travel_agent.version}")

In [ ]:
# Test the travel agent with a data-lookup question
conv_travel = openai_client.conversations.create(
    items=[{"type": "message", "role": "user", "content": "What hotels are available and what are their nightly rates?"}],
)

response_travel = openai_client.responses.create(
    conversation=conv_travel.id,
    extra_body={"agent_reference": {"name": travel_agent.name, "type": "agent_reference"}},
)
print("--- AGENT (File Search) ---")
print(response_travel.output_text)

In [ ]:
# Test with a budget calculation (should trigger Code Interpreter)
openai_client.conversations.items.create(
    conversation_id=conv_travel.id,
    items=[{"type": "message", "role": "user", "content": "Compare the total cost of a 3-night stay at each hotel including the cheapest flight option."}],
)

response_budget = openai_client.responses.create(
    conversation=conv_travel.id,
    extra_body={"agent_reference": {"name": travel_agent.name, "type": "agent_reference"}},
)
print("--- AGENT (Budget Calculation) ---")
print(response_budget.output_text)

---
## Part 3: Azure AI Search — Production-Grade RAG

Connect an agent to Azure AI Search for enterprise-scale retrieval.

> **Note:** This requires `AI_SEARCH_PROJECT_CONNECTION_ID` and `AI_SEARCH_INDEX_NAME` in your `.env`. If not set, skip this section.

In [ ]:
# Configure the Azure AI Search tool
assert AI_SEARCH_PROJECT_CONNECTION_ID, "AI_SEARCH_PROJECT_CONNECTION_ID not set — skip this section"
assert AI_SEARCH_INDEX_NAME, "AI_SEARCH_INDEX_NAME not set — skip this section"

ai_search_tool = AzureAISearchTool(
    azure_ai_search=AzureAISearchToolResource(
        indexes=[
            AISearchIndexResource(
                project_connection_id=AI_SEARCH_PROJECT_CONNECTION_ID,
                index_name=AI_SEARCH_INDEX_NAME,
                query_type=AzureAISearchQueryType.SIMPLE,
            ),
        ]
    )
)

print(f"✅ AI Search tool configured for index: {AI_SEARCH_INDEX_NAME}")

In [ ]:
# Create an agent with the AI Search tool
news_agent_search = project_client.agents.create_version(
    agent_name="NewsAgentWithSearch",
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT_NAME,
        instructions=(
            "You are a travel news assistant connected to a search index of news and event documents. "
            "Use Azure AI Search to find relevant news articles, events, and travel advisories. "
            "Always ground your responses in search results and cite sources when providing information. "
            "If no relevant results are found, clearly state that no matching information was found in the index."
        ),
        tools=[ai_search_tool],
    ),
)
print(f"✅ Agent with AI Search created — name: {news_agent_search.name}, version: {news_agent_search.version}")

In [ ]:
# Test the AI Search agent
conv_search = openai_client.conversations.create(
    items=[{"type": "message", "role": "user", "content": "Are there any upcoming music festivals in Barcelona?"}],
)

response_search = openai_client.responses.create(
    conversation=conv_search.id,
    extra_body={"agent_reference": {"name": news_agent_search.name, "type": "agent_reference"}},
)
print("--- AGENT (AI Search) ---")
print(response_search.output_text)

---
## Part 4: Grounding Comparison

Compare an agent WITH file search tools vs one WITHOUT, on the same question.

In [ ]:
test_question = "What is the nightly rate for the cheapest hotel available?"

# --- Agent WITHOUT tools (will hallucinate) ---
bare_agent = project_client.agents.create_version(
    agent_name="BareAgent-Test",
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT_NAME,
        instructions="You are a travel assistant. Answer questions about hotels.",
    ),
)

conv_bare = openai_client.conversations.create(
    items=[{"type": "message", "role": "user", "content": test_question}],
)
resp_bare = openai_client.responses.create(
    conversation=conv_bare.id,
    extra_body={"agent_reference": {"name": bare_agent.name, "type": "agent_reference"}},
)

print("=" * 60)
print("❌ AGENT WITHOUT TOOLS (likely hallucinated):")
print("=" * 60)
print(resp_bare.output_text)

# --- Agent WITH File Search (grounded) ---
conv_grounded = openai_client.conversations.create(
    items=[{"type": "message", "role": "user", "content": test_question}],
)
resp_grounded = openai_client.responses.create(
    conversation=conv_grounded.id,
    extra_body={"agent_reference": {"name": travel_agent.name, "type": "agent_reference"}},
)

print("\n")
print("=" * 60)
print("✅ AGENT WITH FILE SEARCH (grounded in data):")
print("=" * 60)
print(resp_grounded.output_text)

print("\n\n💡 Notice the difference? The grounded agent cites real data.")
print("   The ungrounded agent makes up plausible-sounding prices.")

# Cleanup the bare test agent
project_client.agents.delete_version(agent_name=bare_agent.name, agent_version=bare_agent.version)
openai_client.conversations.delete(conversation_id=conv_bare.id)
openai_client.conversations.delete(conversation_id=conv_grounded.id)

---
## Cleanup

⚠️ **Do NOT delete your agents yet** — they are needed for later challenges.

In [ ]:
# ONLY run this if you need to start fresh — uncomment below:

# openai_client.conversations.delete(conversation_id=conversation.id)
# openai_client.conversations.delete(conversation_id=conv_travel.id)
# project_client.agents.delete_version(agent_name=news_agent_with_tools.name, agent_version=news_agent_with_tools.version)
# project_client.agents.delete_version(agent_name=travel_agent.name, agent_version=travel_agent.version)

print("ℹ️ Agents preserved for later challenges.")
print(f"  NewsAgentWithTools: {news_agent_with_tools.name} v{news_agent_with_tools.version}")
print(f"  TravelAgentWithSearch: {travel_agent.name} v{travel_agent.version}")